# Tiki Plan B — Repair vòng 2 (per-model)

Dành cho 2.109 dòng vẫn chưa dịch sau vòng 1: model copy nguyên tiếng Việt
hoặc JSON bị cắt.

- Prompt **brand-aware**: chỉ giữ proper noun/brand/model code, còn danh từ
  tiếng Việt phải dịch kể cả khi đứng cạnh brand.
- Probe 60 dòng/model với 3 biến thể, chọn tốt nhất.
- 2 lần thử/dòng, lấy kết quả đầu tiên đạt.
- Input: `tiki-bt-repair-input-v2` (0.6B 1.315 | 1.7B 619 | 4B 40 | 8B 39).


In [ ]:
import json, os, re, time, glob
from pathlib import Path
import torch

print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('vram_gb', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
import transformers
print('transformers', transformers.__version__)
from transformers import AutoTokenizer, AutoModelForCausalLM
print('Qwen3ForCausalLM OK')
try:
    import kagglehub
    print('kagglehub', kagglehub.__version__)
except Exception as error:
    kagglehub = None
    print('kagglehub unavailable:', error)


In [ ]:
INPUT_DIR = Path('/kaggle/input/tiki-bt-repair-input-v2')
if not (INPUT_DIR / 'repair_input_v2.parquet').exists():
    hits = glob.glob('/kaggle/input/**/repair_input_v2.parquet', recursive=True)
    INPUT_DIR = Path(hits[0]).parent if hits else INPUT_DIR
print('INPUT_DIR', INPUT_DIR)

OUTPUT_ROOT = Path('/kaggle/working/repair_outputs_v2')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODELS = ['qwen3-0.6b', 'qwen3-1.7b', 'qwen3-4b', 'qwen3-8b']
MODEL_SOURCES = {
    'qwen3-0.6b': 'qwen-lm/qwen-3/Transformers/0.6b/1',
    'qwen3-1.7b': 'qwen-lm/qwen-3/Transformers/1.7b/1',
    'qwen3-4b':   'qwen-lm/qwen-3/Transformers/4b/1',
    'qwen3-8b':   'qwen-lm/qwen-3/Transformers/8b/1',
}
EXPECTED_SHAPE = {
    'qwen3-0.6b': (1024, 3072, 28),
    'qwen3-1.7b': (2048, 6144, 28),
    'qwen3-4b':   (2560, 9728, 36),
    'qwen3-8b':   (4096, 12288, 36),
}
BATCH_BY_MODEL = {'qwen3-0.6b': 16, 'qwen3-1.7b': 16, 'qwen3-4b': 12, 'qwen3-8b': 8}

MAX_NEW_TOKENS = 2048
MAX_INPUT_TOKENS = 1280
PROBE_SIZE = 60
PROMPT_VERSION = 'plan_b_repair_v5'
TIME_BUDGET_MIN = 300
DEADLINE = time.time() + TIME_BUDGET_MIN * 60
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('time_budget_min', TIME_BUDGET_MIN)


In [ ]:
BRAND_AWARE = '''You are a professional Vietnamese-to-Simplified Chinese e-commerce translator.
The field "source_text" is a product title or attribute value. Translate ALL Vietnamese words into Simplified Chinese.
Rules:
- Keep only proper nouns, brand names and Latin model codes unchanged (e.g. "Cozy Icetea", "iPhone 13", "La Roche-Posay").
- Vietnamese common nouns MUST be translated even when they stand next to a brand: "Chanh" -> 柠檬, "Tui"/"Túi" -> 袋装, "Hộp" -> 盒, "Ly" -> 杯, "Khóa" -> 锁.
- Keep numbers, sizes and units unchanged.
- The output MUST contain Chinese characters. Never return the source unchanged.
Return JSON only: {"translation": "..."}

Examples:
{"source_text": "Cozy Icetea Chanh 400gr Túi"} -> {"translation": "Cozy Icetea 柠檬味 400克 袋装"}
{"source_text": "Khóa Thắng Đĩa Master Lock 8303 DPS"} -> {"translation": "Master Lock 8303 DPS 碟刹锁"}
{"source_text": "Chì mày vặn 1 đầu Camelo"} -> {"translation": "Camelo 单头旋转眉笔"}
{"source_text": "Xí Ngầu Trong Suốt Liên Hiệp Thành"} -> {"translation": "联盟成 透明骰子"}
{"source_text": "iPhone 15 Pro Max"} -> {"translation": "iPhone 15 Pro Max"}'''

V4_FEWSHOT = '''You are a professional Vietnamese-to-Simplified Chinese e-commerce translator.
The field "source_text" is the VALUE that must be translated.
"key_vi"/"key_zh" describe what kind of attribute it is: use them as context only,
never translate or copy the key into the output.
Rules:
- Output MUST be written in Simplified Chinese characters.
- Never copy Vietnamese words, except brand names and model codes.
- Translate even a single noun (e.g. "Hổ" -> "虎").
- Keep numbers, units, sizes, SKUs and Latin model codes unchanged.
Return JSON only: {"translation": "..."}

Examples:
{"source_text": "Hổ", "key_vi": "Họa tiết"} -> {"translation": "虎"}
{"source_text": "Xám", "key_vi": "Mầu Sắc"} -> {"translation": "灰色"}
{"source_text": "Số 4: 5L", "key_vi": "Kích cỡ"} -> {"translation": "4号：5升"}
{"source_text": "Z Fold 3", "key_vi": "Size"} -> {"translation": "Z Fold 3"}
{"source_text": "Đen viền xám", "key_vi": "Màu sắc"} -> {"translation": "黑边灰色"}'''

VARIANTS = {
    'brand_aware': (BRAND_AWARE, True),
    'no_key': (BRAND_AWARE, False),
    'v4_fewshot': (V4_FEWSHOT, True),
}
CJK_RE = re.compile(r'[\u4e00-\u9fff]')
VI_RE = re.compile(r'[ăâêôơưđ]|[áàảãạấầẩẫậắằẳẵặéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]', re.IGNORECASE)

def build_prompt(tokenizer, row, system, with_key=True):
    payload = {'source_text': str(row.get('source_text_bt') or '')}
    if with_key:
        payload['key_vi'] = str(row.get('key_vi') or '')
        payload['key_zh'] = str(row.get('key_zh_effective') or '')
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': json.dumps(payload, ensure_ascii=False)},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False,
                                             add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_translation(text):
    raw = (text or '').strip()
    start, end = raw.find('{'), raw.rfind('}')
    if start >= 0 and end > start:
        try:
            payload = json.loads(raw[start:end + 1])
            if isinstance(payload, dict) and str(payload.get('translation') or '').strip():
                return str(payload['translation']).strip(), True
        except json.JSONDecodeError:
            pass
    return '', False

def good_translation(translation):
    text = (translation or '').strip()
    if not text:
        return False
    if CJK_RE.search(text):
        return True
    return not VI_RE.search(text)


In [ ]:
def find_model_dir(model_key):
    if kagglehub is not None:
        try:
            path = kagglehub.model_download(MODEL_SOURCES[model_key])
            print(model_key, 'kagglehub ->', path)
            return path
        except Exception as error:
            print(model_key, 'kagglehub failed:', type(error).__name__, error)

    expected = EXPECTED_SHAPE[model_key]
    for config_path in Path('/kaggle/input').rglob('config.json'):
        try:
            payload = json.loads(config_path.read_text(encoding='utf-8'))
        except Exception:
            continue
        shape = (payload.get('hidden_size'), payload.get('intermediate_size'),
                 payload.get('num_hidden_layers'))
        if shape == expected:
            print(model_key, 'fallback ->', config_path.parent)
            return str(config_path.parent)
    raise FileNotFoundError('Không tìm thấy weights cho ' + model_key)

def load_model(model_dir):
    kwargs = dict(trust_remote_code=True, local_files_only=True, low_cpu_mem_usage=True)
    try:
        return AutoModelForCausalLM.from_pretrained(model_dir, dtype=torch.bfloat16, **kwargs)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.bfloat16, **kwargs)


In [ ]:
import pandas as pd

repair = pd.read_parquet(INPUT_DIR / 'repair_input_v2.parquet')
print('repair rows', len(repair))
print(repair.groupby('model_key').size().to_dict())


In [ ]:
def generate_batch(model, tokenizer, rows, system, with_key, batch_size):
    records = []
    for start in range(0, len(rows), batch_size):
        batch = rows.iloc[start:start + batch_size].to_dict('records')
        prompts = [build_prompt(tokenizer, row, system, with_key) for row in batch]
        enc = tokenizer(prompts, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_INPUT_TOKENS).to(DEVICE)
        started = time.time()
        with torch.inference_mode():
            output = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                    num_beams=1, eos_token_id=tokenizer.eos_token_id,
                                    pad_token_id=tokenizer.pad_token_id)
        latency_ms = (time.time() - started) * 1000 / max(1, len(batch))
        generated = output[:, enc['input_ids'].shape[1]:]
        texts = tokenizer.batch_decode(generated, skip_special_tokens=True)
        n_new = [int((row != tokenizer.pad_token_id).sum()) for row in generated]
        for row, raw, ntokens in zip(batch, texts, n_new):
            value, ok = parse_translation(raw)
            records.append({
                'model_key': row['model_key'],
                'cache_key': str(row['cache_key']),
                'source_text_bt': row.get('source_text_bt'),
                'target_text_bt': value,
                'raw_output': raw,
                'json_ok': bool(ok),
                'good': bool(good_translation(value)),
                'n_new_tokens': int(ntokens),
                'latency_ms': round(latency_ms, 2),
            })
        if (start // batch_size) % 10 == 0 or start + len(batch) >= len(rows):
            print('  batch', start + len(batch), '/', len(rows), flush=True)
    return records


In [ ]:
def probe(model, tokenizer, rows, batch_size):
    probe = rows.assign(_len=rows['source_text_bt'].fillna('').str.len()).sort_values('_len')
    if len(probe) > PROBE_SIZE:
        idx = [int(i * (len(probe) - 1) / (PROBE_SIZE - 1)) for i in range(PROBE_SIZE)]
        probe = probe.iloc[idx]
    scores = {}
    for name, (system, with_key) in VARIANTS.items():
        recs = generate_batch(model, tokenizer, probe, system, with_key, max(1, batch_size // 2))
        rate = sum(r['good'] for r in recs) / max(1, len(recs))
        scores[name] = round(rate, 3)
        print('probe', name, rate)
    return scores


In [ ]:
def run_model(model_key):
    out_dir = OUTPUT_ROOT / model_key
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / 'repair_v2.parquet'
    rows = repair[repair['model_key'] == model_key].reset_index(drop=True)
    if rows.empty:
        print('SKIP', model_key)
        return {'model_key': model_key, 'n_rows': 0}

    model_dir = find_model_dir(model_key)
    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True, local_files_only=True)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(model_key, 'loading weights...')
    model = load_model(model_dir).to(DEVICE).eval()
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    print(model_key, 'loaded | vram_alloc_gb', round(torch.cuda.memory_allocated() / 1e9, 1))

    batch_size = BATCH_BY_MODEL[model_key]
    started = time.time()
    scores = probe(model, tokenizer, rows, batch_size)
    order = sorted(scores, key=lambda k: scores[k], reverse=True)
    print(model_key, 'variant order', order, scores)

    frame = pd.DataFrame()
    stopped = 'completed'
    for start in range(0, len(rows), 500):
        if time.time() >= DEADLINE:
            stopped = 'time_budget'
            break
        chunk = rows.iloc[start:start + 500].reset_index(drop=True)
        pending = chunk
        results = []
        for attempt, variant in enumerate(order[:2], 1):
            if pending.empty:
                break
            system, with_key = VARIANTS[variant]
            recs = generate_batch(model, tokenizer, pending, system, with_key, batch_size)
            for rec in recs:
                rec['attempt_variant'] = variant
                rec['attempt'] = attempt
            good_keys = {r['cache_key'] for r in recs if r['good']}
            results.extend([r for r in recs if r['good']])
            pending = pending[~pending['cache_key'].astype(str).isin(good_keys)].reset_index(drop=True)

        # dòng vẫn hỏng: giữ kết quả lần 2 để audit
        if not pending.empty:
            system, with_key = VARIANTS[order[-1]]
            recs = generate_batch(model, tokenizer, pending, system, with_key, batch_size)
            for rec in recs:
                rec['attempt_variant'] = order[-1]
                rec['attempt'] = 3
            results.extend(recs)

        frame = pd.concat([frame, pd.DataFrame(results)], ignore_index=True)
        frame.to_parquet(out_file, index=False)
        n_good = int(frame['good'].sum())
        print(model_key, 'saved', len(frame), '/', len(rows),
              'good', n_good, f'({n_good / len(frame) * 100:.1f}%)', flush=True)

    state = {
        'model_key': model_key,
        'n_rows': int(len(frame)),
        'n_good': int(frame['good'].sum()),
        'good_rate': round(float(frame['good'].mean()), 4),
        'variant_scores': scores,
        'stopped': stopped,
        'elapsed_min': round((time.time() - started) / 60, 1),
        'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    (out_dir / 'run_state.json').write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')
    print(model_key, 'DONE', json.dumps(state, ensure_ascii=False))
    del model
    torch.cuda.empty_cache()
    return state


In [ ]:
summary = []
for model_key in MODELS:
    if time.time() >= DEADLINE:
        print('Hết time budget, dừng trước', model_key)
        break
    summary.append(run_model(model_key))

payload = {'prompt_version': PROMPT_VERSION, 'runs': summary,
           'written_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
Path('/kaggle/working/repair_summary_v2.json').write_text(
    json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(payload, ensure_ascii=False, indent=2))
